#### Import

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import lars_path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas_datareader.data as web
import warnings
import os
from src.config import START_DATE, END_DATE, N_VARS_LASSO
from src.utils import select_exact_k_lars, get_stars, get_ar1_innovations
from src.data_loader import load_topics, get_ken_french_49_daily, transform_fred_md

warnings.filterwarnings('ignore')
print(f"Environment loaded. Analysis period: {START_DATE} to {END_DATE}")

In [ ]:
theta, labels_map = load_topics('data')

if theta is not None:
    print(f"Topics loaded: {theta.shape[1]} topics over {len(theta)} months.")
else:
    print("Error")

#### Section 1. News attention and economic activity 

##### 1.1. Macroeconomic times series    

###### Configuration and data

In [ ]:
MIN_TRAIN_SIZE = 120
N_VARS = 5

NAME_MAPPING = {'IP': 'Industrial Production Growth','Emp': 'Employment Growth','MktRet': 'Market Returns','MktVol': 'Market Volatility'}

TICKERS = {'INDPRO': 'IP', 'PAYEMS': 'Emp', 'SPASTT01USM661N': 'MktRet', 'VIXCLS': 'MktVol'}

try:
    fred_raw = web.DataReader(list(TICKERS.keys()), 'fred', start='1980-01-01')
    fred_m = fred_raw.resample('MS').mean()
    data = pd.DataFrame(index=fred_m.index)
    
    data['IP'] = np.log(fred_m['INDPRO']).diff() * 100
    data['Emp'] = np.log(fred_m['PAYEMS']).diff() * 100
    data['MktRet'] = np.log(fred_m['SPASTT01USM661N']).diff() * 100
    data['MktVol'] = fred_m['VIXCLS'] 
    data = data.dropna()
except:
    print("Error downloading FRED.")

common = theta.index.intersection(data.index)
X = theta.loc[common]
Y = data.loc[common]

def get_stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.1: return "*"
    return ""

# Local selection function
def select_exact_k_variables_local(X, y, k=5):
    alphas, active, coefs = lars_path(X.values, y.values, method='lasso')
    n_active = np.count_nonzero(coefs, axis=0)
    step_idx = np.where(n_active == k)[0]
    if len(step_idx) > 0: chosen_step = step_idx[0]
    else: chosen_step = np.argmin(np.abs(n_active - k))
    active_indices = np.where(coefs[:, chosen_step] != 0)[0]
    return X.columns[active_indices].tolist() # Returns list of names

###### Analysis function

In [ ]:
def run_forecast_analysis(short_name, full_name, y_target, X_features):    
    # 1. Standardization
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    X_std = pd.DataFrame(scaler_X.fit_transform(X_features), index=X_features.index, columns=X_features.columns)
    y_std = pd.Series(scaler_y.fit_transform(y_target.values.reshape(-1,1)).flatten(), index=y_target.index)
    
    # A. IN-SAMPLE ANALYSIS
    top_5_vars = select_exact_k_variables_local(X_std, y_std, k=N_VARS)
    X_ols = sm.add_constant(X_std[top_5_vars])
    model = sm.OLS(y_std, X_ols).fit()
    r2_in = model.rsquared
    
    # B. OUT-OF-SAMPLE ANALYSIS
    trues_oos = []
    preds_oos = []
    hist_means = []
    
    for i in range(MIN_TRAIN_SIZE, len(y_std)):
        y_train = y_std.iloc[:i]
        X_train = X_std.iloc[:i]
        
        y_test_pt = y_std.iloc[i]
        X_test_row = X_std.iloc[[i]]
        
        # Selection on Train
        vars_t = select_exact_k_variables_local(X_train, y_train, k=N_VARS)
        
        # Fit & Predict
        reg = LinearRegression().fit(X_train[vars_t], y_train)
        pred = reg.predict(X_test_row[vars_t])[0]
        
        preds_oos.append(pred)
        trues_oos.append(y_test_pt)
        hist_means.append(y_train.mean())

    # R2
    mse_model = np.mean((np.array(trues_oos) - np.array(preds_oos))**2)
    mse_bench = np.mean((np.array(trues_oos) - np.array(hist_means))**2)
    r2_out = 1 - (mse_model / mse_bench)
    
    
    # C. TABLES
    res = pd.DataFrame({
        'Coeff': model.params[top_5_vars], 
        'Pval': model.pvalues[top_5_vars]
    })
    # Mapping
    res['Topic'] = [labels_map.get(str(v), v) if str(v) in labels_map else labels_map.get(v, v) for v in res.index]
    res = res.sort_values(by='Coeff', key=abs, ascending=False)
    
    print(f"\n{'-'*75}")
    print(f"TABLE : {full_name.upper()}")
    print(f"{'-'*75}")
    print(f"{'Topic':<35} | {'Coeff.':>7} | {'P-val':>8} | {'Sig'}")
    print(f"{'-'*75}")
    
    for _, row in res.iterrows():
        star = get_stars(row['Pval'])
        print(f"{row['Topic']:<35} | {row['Coeff']:>7.2f} | {row['Pval']:>8.4f} | {star}")
        
    print(f"{'-'*75}")
    print(f"In-Sample R2     : {r2_in:.2f}")
    print(f"Out-of-Sample R2 : {r2_out:.2f}")
    print(f"{'-'*75}")
    
    
    # D. GRAPHS
    vars_final = select_exact_k_variables_local(X_std, y_std, k=N_VARS)
    reg_final = LinearRegression().fit(X_std[vars_final], y_std)
    preds_in_sample = reg_final.predict(X_std[vars_final])

    dates_oos = y_std.index[MIN_TRAIN_SIZE:] 
    preds_oos_aligned = pd.Series(preds_oos, index=dates_oos)

    plt.figure(figsize=(12, 5))
    plt.plot(y_std.index, y_std, 'k-', lw=1, alpha=0.3, label='Actual Data')
    plt.plot(y_std.index, preds_in_sample, 'b--', lw=1, alpha=0.8, label='In-Sample Fit (Explain)')
    plt.plot(dates_oos, preds_oos_aligned, 'r-', lw=1.5, label='Out-of-Sample Forecast')
    
    plt.axvline(y_std.index[MIN_TRAIN_SIZE], color='gray', linestyle=':', label='OOS Start (10y)')
    plt.title(f"{full_name}: Topic Model Performance")
    plt.legend()
    plt.tight_layout()
    
    plt.show()

###### Execution

In [ ]:
for short_col, full_name in NAME_MAPPING.items():
    if short_col in Y.columns:
        run_forecast_analysis(short_col, full_name, Y[short_col], X)

In [ ]:
# 1. CONFIGURATION & DATA
MIN_TRAIN_SIZE = 120
N_VARS = 5

NAME_MAPPING = {'IP': 'Industrial Production Growth','Emp': 'Employment Growth','MktRet': 'Market Returns','MktVol': 'Market Volatility'}

TICKERS = {'INDPRO': 'IP', 'PAYEMS': 'Emp', 'SPASTT01USM661N': 'MktRet', 'VIXCLS': 'MktVol'}

try:
    fred_raw = web.DataReader(list(TICKERS.keys()), 'fred', start='1980-01-01')
    fred_m = fred_raw.resample('MS').mean()
    data = pd.DataFrame(index=fred_m.index)
    
    data['IP'] = np.log(fred_m['INDPRO']).diff() * 100
    data['Emp'] = np.log(fred_m['PAYEMS']).diff() * 100
    data['MktRet'] = np.log(fred_m['SPASTT01USM661N']).diff() * 100
    data['MktVol'] = fred_m['VIXCLS'] 
    data = data.dropna()
except:
    print("Error downloading FRED.")

common = theta.index.intersection(data.index)
X = theta.loc[common]
Y = data.loc[common]

def get_stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.1: return "*"
    return ""

# Local selection function
def select_exact_k_variables_local(X, y, k=5):
    alphas, active, coefs = lars_path(X.values, y.values, method='lasso')
    n_active = np.count_nonzero(coefs, axis=0)
    step_idx = np.where(n_active == k)[0]
    if len(step_idx) > 0: chosen_step = step_idx[0]
    else: chosen_step = np.argmin(np.abs(n_active - k))
    active_indices = np.where(coefs[:, chosen_step] != 0)[0]
    return X.columns[active_indices].tolist() # Returns list of names

# 2. ANALYSIS FUNCTION
def run_forecast_analysis(short_name, full_name, y_target, X_features):
    print(f"\nProcessing: {full_name}...")
    
    # 1. Standardization
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    X_std = pd.DataFrame(scaler_X.fit_transform(X_features), index=X_features.index, columns=X_features.columns)
    y_std = pd.Series(scaler_y.fit_transform(y_target.values.reshape(-1,1)).flatten(), index=y_target.index)
    
    # A. IN-SAMPLE ANALYSIS
    top_5_vars = select_exact_k_variables_local(X_std, y_std, k=N_VARS)
    X_ols = sm.add_constant(X_std[top_5_vars])
    model = sm.OLS(y_std, X_ols).fit()
    r2_in = model.rsquared
    
    # B. OUT-OF-SAMPLE ANALYSIS
    trues_oos = []
    preds_oos = []
    hist_means = []
    
    for i in range(MIN_TRAIN_SIZE, len(y_std)):
        y_train = y_std.iloc[:i]
        X_train = X_std.iloc[:i]
        
        y_test_pt = y_std.iloc[i]
        X_test_row = X_std.iloc[[i]]
        
        # Selection on Train
        vars_t = select_exact_k_variables_local(X_train, y_train, k=N_VARS)
        
        # Fit & Predict
        reg = LinearRegression().fit(X_train[vars_t], y_train)
        pred = reg.predict(X_test_row[vars_t])[0]
        
        preds_oos.append(pred)
        trues_oos.append(y_test_pt)
        hist_means.append(y_train.mean())

    # R2
    mse_model = np.mean((np.array(trues_oos) - np.array(preds_oos))**2)
    mse_bench = np.mean((np.array(trues_oos) - np.array(hist_means))**2)
    r2_out = 1 - (mse_model / mse_bench)
    
    
    # C. TABLES
    res = pd.DataFrame({
        'Coeff': model.params[top_5_vars], 
        'Pval': model.pvalues[top_5_vars]
    })
    # Mapping
    res['Topic'] = [labels_map.get(str(v), v) if str(v) in labels_map else labels_map.get(v, v) for v in res.index]
    res = res.sort_values(by='Coeff', key=abs, ascending=False)
    
    print(f"\n{'-'*75}")
    print(f"TABLE : {full_name.upper()}")
    print(f"{'-'*75}")
    print(f"{'Topic':<35} | {'Coeff.':>7} | {'P-val':>8} | {'Sig'}")
    print(f"{'-'*75}")
    
    for _, row in res.iterrows():
        star = get_stars(row['Pval'])
        print(f"{row['Topic']:<35} | {row['Coeff']:>7.2f} | {row['Pval']:>8.4f} | {star}")
        
    print(f"{'-'*75}")
    print(f"In-Sample R2     : {r2_in:.2f}")
    print(f"Out-of-Sample R2 : {r2_out:.2f}")
    print(f"{'-'*75}")
    
    
    # D. GRAPHS
    vars_final = select_exact_k_variables_local(X_std, y_std, k=N_VARS)
    reg_final = LinearRegression().fit(X_std[vars_final], y_std)
    preds_in_sample = reg_final.predict(X_std[vars_final])

    dates_oos = y_std.index[MIN_TRAIN_SIZE:] 
    preds_oos_aligned = pd.Series(preds_oos, index=dates_oos)

    plt.figure(figsize=(12, 5))
    plt.plot(y_std.index, y_std, 'k-', lw=1, alpha=0.3, label='Actual Data')
    plt.plot(y_std.index, preds_in_sample, 'b--', lw=1, alpha=0.8, label='In-Sample Fit (Explain)')
    plt.plot(dates_oos, preds_oos_aligned, 'r-', lw=1.5, label='Out-of-Sample Forecast')
    
    plt.axvline(y_std.index[MIN_TRAIN_SIZE], color='gray', linestyle=':', label='OOS Start (10y)')
    plt.title(f"{full_name}: Topic Model Performance")
    plt.legend()
    plt.tight_layout()
    
    plt.show()


# 3. EXECUTION
for short_col, full_name in NAME_MAPPING.items():
    if short_col in Y.columns:
        run_forecast_analysis(short_col, full_name, Y[short_col], X)

##### 1.2. Stock returns

###### Configuration and data

In [ ]:
TARGET_START = '1984-01-01'
TARGET_END = '2017-06-01'

CFNAI_IDS = {'CFNAI': 'CFNAI', 'PANDI': 'Prod_Inc', 'EUANDH': 'Emp_Hrs', 'CANDH': 'Cons_Hous', 'SOANDI': 'Sales_Ord'}
FRED_MD_IDS = ['INDPRO', 'PAYEMS', 'UNRATE', 'HOUST', 'PPIACO', 'PCEPI', 'M2SL', 'FEDFUNDS']

# 1. DOWNLOAD & DIAGNOSTIC

# A. Topics (Security)
try:
    if 'theta' not in locals(): theta, labels_map = load_topics('data')
except: pass 

# B. Target (S&P 500)
try:
    sp500 = web.DataReader('SPASTT01USM661N', 'fred', start='1950-01-01')
    mkt_ret = np.log(sp500).diff() * 100
    mkt_ret.columns = ['MktRet']
    mkt_ret = mkt_ret.dropna()
except: print("Error Target.")

# C. Benchmarks
try:
    cfnai_raw = web.DataReader(list(CFNAI_IDS.keys()), 'fred', start='1960').rename(columns=CFNAI_IDS).resample('MS').mean().dropna(axis=1, how='all')
    cfnai_innov = get_ar1_innovations(cfnai_raw)
except: cfnai_innov = pd.DataFrame()

try:
    fred_list = [web.DataReader(c, 'fred', start='1960').resample('MS').mean() for c in FRED_MD_IDS]
    fred_raw = pd.concat(fred_list, axis=1)
    fred_stat = transform_fred_md(fred_raw)
    fred_innov = get_ar1_innovations(fred_stat)
except: fred_innov = pd.DataFrame()

# 2. ALIGNMENT
common = mkt_ret.index.intersection(theta.index)
common = common[(common >= TARGET_START) & (common <= TARGET_END)]
final_idx = common.intersection(cfnai_innov.index).intersection(fred_innov.index)

print(f"Common period: {len(final_idx)} months ({final_idx.min().date()} -> {final_idx.max().date()})")

y = mkt_ret.loc[final_idx]
X_topics = theta.loc[final_idx]
X_cfnai = cfnai_innov.loc[final_idx]
X_fred = fred_innov.loc[final_idx]

def get_stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.1: return "*"
    return ""

###### Analysis function

In [ ]:
def run_standardized_analysis(y_target, X_news, X_bench, bench_name):
    if X_bench.empty: return

    # Standardization
    scaler = StandardScaler()
    y_std = pd.Series(scaler.fit_transform(y_target.values.reshape(-1, 1)).flatten(), index=y_target.index)
    X_news_std = pd.DataFrame(scaler.fit_transform(X_news), index=X_news.index, columns=X_news.columns)
    X_bench_std = pd.DataFrame(scaler.fit_transform(X_bench), index=X_bench.index, columns=X_bench.columns)

    # 1. Benchmark R2
    if X_bench.shape[1] <= 5:
        model_b = sm.OLS(y_std, sm.add_constant(X_bench_std)).fit()
    else:
        # Lasso Benchmark
        vars_b = select_exact_k_lars(X_bench_std, y_std, k=5)
        # Automatic handling (Names or Indices)
        if len(vars_b) > 0 and isinstance(vars_b[0], (int, np.integer)): 
            vars_b = X_bench_std.columns[vars_b]
        model_b = sm.OLS(y_std, sm.add_constant(X_bench_std[vars_b])).fit()
        
    bench_r2 = model_b.rsquared

    # 2. Full Model
    X_pool = pd.concat([X_news_std, X_bench_std], axis=1)
    
    # Lasso Selection
    selected_vars = select_exact_k_lars(X_pool, y_std, k=5)
    
    # --- CRITICAL FIX: TYPE DETECTION ---
    if len(selected_vars) > 0 and isinstance(selected_vars[0], (int, np.integer)):
        selected_vars = X_pool.columns[selected_vars]
    
    # Final Model
    full_model = sm.OLS(y_std, sm.add_constant(X_pool[selected_vars])).fit()
    
    # Display
    print(f"\n{'-'*75}")
    print(f"TABLE 2: TOPIC MODEL & {bench_name}")
    print(f"{'-'*75}")
    print(f"{'Topic / Variable':<35} | {'Coeff.':>7} | {'P-val':>8} | {'Sig'}")
    print(f"{'-'*75}")
    
    res = pd.DataFrame({
        'Name': selected_vars,
        'Coeff': full_model.params[selected_vars],
        'Pval': full_model.pvalues[selected_vars]
    }).sort_values(by='Coeff', key=abs, ascending=False)
    
    for _, row in res.iterrows():
        var_name = str(row['Name'])
        # Display name mapping
        if var_name in labels_map:
            display_name = labels_map[var_name]
        elif var_name in CFNAI_IDS.values():
            display_name = f"[CFNAI] {var_name}"
        else:
            display_name = f"[MACRO] {var_name}"
            
        star = get_stars(row['Pval'])
        print(f"{display_name:<35} | {row['Coeff']:>7.2f} | {row['Pval']:>8.4f} | {star}")
        
    print(f"{'-'*75}")
    print(f"Full R2      : {full_model.rsquared:.2f}")
    print(f"Benchmark R2 : {bench_r2:.2f}")
    print(f"{'-'*75}")

###### Execution

In [ ]:
run_standardized_analysis(y, X_topics, X_cfnai, "CFNAI")
run_standardized_analysis(y, X_topics, X_fred, "FRED-MD")

In [ ]:
print("SECTION 4.2: RECONSTRUCTING STOCK RETURNS (Final Robust)")


# 0. SPECIFIC CONFIGURATION

TARGET_START = '1984-01-01'
TARGET_END = '2017-06-01'

CFNAI_IDS = {'CFNAI': 'CFNAI', 'PANDI': 'Prod_Inc', 'EUANDH': 'Emp_Hrs', 'CANDH': 'Cons_Hous', 'SOANDI': 'Sales_Ord'}
FRED_MD_IDS = ['INDPRO', 'PAYEMS', 'UNRATE', 'HOUST', 'PPIACO', 'PCEPI', 'M2SL', 'FEDFUNDS']


# 1. DOWNLOAD & DIAGNOSTIC

print("1. LOADING DATA & DIAGNOSTIC...")

# A. Topics (Security)
try:
    if 'theta' not in locals(): theta, labels_map = load_topics('data')
except: pass 

# B. Target (S&P 500)
try:
    sp500 = web.DataReader('SPASTT01USM661N', 'fred', start='1950-01-01')
    mkt_ret = np.log(sp500).diff() * 100
    mkt_ret.columns = ['MktRet']
    mkt_ret = mkt_ret.dropna()
except: print("Error Target.")

# C. Benchmarks
try:
    cfnai_raw = web.DataReader(list(CFNAI_IDS.keys()), 'fred', start='1960').rename(columns=CFNAI_IDS).resample('MS').mean().dropna(axis=1, how='all')
    cfnai_innov = get_ar1_innovations(cfnai_raw)
except: cfnai_innov = pd.DataFrame()

try:
    fred_list = [web.DataReader(c, 'fred', start='1960').resample('MS').mean() for c in FRED_MD_IDS]
    fred_raw = pd.concat(fred_list, axis=1)
    fred_stat = transform_fred_md(fred_raw)
    fred_innov = get_ar1_innovations(fred_stat)
except: fred_innov = pd.DataFrame()

# 2. ALIGNMENT
common = mkt_ret.index.intersection(theta.index)
common = common[(common >= TARGET_START) & (common <= TARGET_END)]
final_idx = common.intersection(cfnai_innov.index).intersection(fred_innov.index)

print(f"Common period: {len(final_idx)} months ({final_idx.min().date()} -> {final_idx.max().date()})")

y = mkt_ret.loc[final_idx]
X_topics = theta.loc[final_idx]
X_cfnai = cfnai_innov.loc[final_idx]
X_fred = fred_innov.loc[final_idx]

# Local stars function
def get_stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.1: return "*"
    return ""

# 3. ANALYSIS FUNCTION (ROBUST + STARS)
def run_standardized_analysis(y_target, X_news, X_bench, bench_name):
    if X_bench.empty: return

    # Standardization
    scaler = StandardScaler()
    y_std = pd.Series(scaler.fit_transform(y_target.values.reshape(-1, 1)).flatten(), index=y_target.index)
    X_news_std = pd.DataFrame(scaler.fit_transform(X_news), index=X_news.index, columns=X_news.columns)
    X_bench_std = pd.DataFrame(scaler.fit_transform(X_bench), index=X_bench.index, columns=X_bench.columns)

    # 1. Benchmark R2
    if X_bench.shape[1] <= 5:
        model_b = sm.OLS(y_std, sm.add_constant(X_bench_std)).fit()
    else:
        # Lasso Benchmark
        vars_b = select_exact_k_lars(X_bench_std, y_std, k=5)
        # Automatic handling (Names or Indices)
        if len(vars_b) > 0 and isinstance(vars_b[0], (int, np.integer)): 
            vars_b = X_bench_std.columns[vars_b]
        model_b = sm.OLS(y_std, sm.add_constant(X_bench_std[vars_b])).fit()
        
    bench_r2 = model_b.rsquared

    # 2. Full Model
    X_pool = pd.concat([X_news_std, X_bench_std], axis=1)
    
    # Lasso Selection
    selected_vars = select_exact_k_lars(X_pool, y_std, k=5)
    
    # --- CRITICAL FIX: TYPE DETECTION ---
    if len(selected_vars) > 0 and isinstance(selected_vars[0], (int, np.integer)):
        selected_vars = X_pool.columns[selected_vars]
    
    # Final Model
    full_model = sm.OLS(y_std, sm.add_constant(X_pool[selected_vars])).fit()
    
    # Display
    print(f"\n{'-'*75}")
    print(f"TABLE 2: TOPIC MODEL & {bench_name}")
    print(f"{'-'*75}")
    print(f"{'Topic / Variable':<35} | {'Coeff.':>7} | {'P-val':>8} | {'Sig'}")
    print(f"{'-'*75}")
    
    res = pd.DataFrame({
        'Name': selected_vars,
        'Coeff': full_model.params[selected_vars],
        'Pval': full_model.pvalues[selected_vars]
    }).sort_values(by='Coeff', key=abs, ascending=False)
    
    for _, row in res.iterrows():
        var_name = str(row['Name'])
        # Display name mapping
        if var_name in labels_map:
            display_name = labels_map[var_name]
        elif var_name in CFNAI_IDS.values():
            display_name = f"[CFNAI] {var_name}"
        else:
            display_name = f"[MACRO] {var_name}"
            
        star = get_stars(row['Pval'])
        print(f"{display_name:<35} | {row['Coeff']:>7.2f} | {row['Pval']:>8.4f} | {star}")
        
    print(f"{'-'*75}")
    print(f"Full R2      : {full_model.rsquared:.2f}")
    print(f"Benchmark R2 : {bench_r2:.2f}")
    print(f"{'-'*75}")

# Run
run_standardized_analysis(y, X_topics, X_cfnai, "CFNAI")
run_standardized_analysis(y, X_topics, X_fred, "FRED-MD")

##### 1.3. Industry volatily 

###### Configuration and data 

In [ ]:
kf_daily = get_ken_french_49_daily()

###### Execution 

In [ ]:
# Monthly volatility
vol_monthly = kf_daily.resample('MS').std()
    
mask = (vol_monthly.index >= START_DATE) & (vol_monthly.index <= END_DATE)
vol_monthly = vol_monthly.loc[mask]
vol_log = np.log(vol_monthly + 1e-6)
pca = PCA(n_components=1)

# Standardization 
vol_log_std = StandardScaler().fit_transform(vol_log)
common_factor = pca.fit_transform(vol_log_std)
    
# Residuals 
vol_ortho = pd.DataFrame(index=vol_log.index, columns=vol_log.columns)
X_factor = sm.add_constant(common_factor)
    
for col in vol_log.columns:
    vol_ortho[col] = sm.OLS(vol_log[col], X_factor).fit().resid
        
# 1. AR(1) innovations 
industry_innov = get_ar1_innovations(vol_ortho)
theta_innov = get_ar1_innovations(theta)
    
# 2. Intersection
common_idx = industry_innov.index.intersection(theta_innov.index)
industry_innov = industry_innov.loc[common_idx]
theta_innov = theta_innov.loc[common_idx]
    
print(f"Volatility alignment: {len(common_idx)} common months.")

# Analysis
# Sectors
TARGET_SECTORS = {'Automotive': ['Autos', 'Auto'], 'Banking': ['Banks', 'Bank'], 'Computers': ['Hrdwr', 'Hardw', 'Comps'], 'Oil': ['Oil', 'Oil  '],'Tobacco': ['Smoke'],'Pharmaceuticals': ['Drugs', 'Drug', 'Pharma']}
    
for name, keywords in TARGET_SECTORS.items():
    col_name = None
    for col in industry_innov.columns:
        clean_col = col.strip().lower()
        if any(k.lower() == clean_col for k in keywords):
            col_name = col
            break

    # Lasso
    y_target = industry_innov[col_name]
            
    # Standardization
    scaler = StandardScaler()
    X_s = pd.DataFrame(scaler.fit_transform(theta_innov), index=theta_innov.index, columns=theta_innov.columns)
    y_s = pd.Series(scaler.fit_transform(y_target.values.reshape(-1,1)).flatten(), index=y_target.index)
            
    # Selection and model
    top_vars = select_exact_k_lars(X_s, y_s, k=5)
    model = sm.OLS(y_s, sm.add_constant(X_s[top_vars])).fit()
            
    print(f"\n{'-'*65}")
    print(f"TABLE 4: {name.upper()} ({col_name}) | R2 = {model.rsquared:.2f}")
    print(f"{'-'*65}")
            
    res = pd.DataFrame({'Coeff': model.params[top_vars],'Pval': model.pvalues[top_vars]})
            
    res['Topic'] = [labels_map.get(str(v), v) if str(v) in labels_map else labels_map.get(v, v) for v in res.index]
    res = res.reindex(res['Coeff'].abs().sort_values(ascending=False).index)
            
    # Pretty print loop
    for idx, row in res.iterrows():
        t_name = row['Topic']
        coeff = row['Coeff']
        stars = get_stars(row['Pval'])
            
        print(f"{t_name:<35} | {coeff:>6.2f} {stars}")
    print(f"{'-'*65}")

In [ ]:
kf_daily = get_ken_french_49_daily()

if kf_daily is not None:
    # Calculate Monthly Volatility
    vol_monthly = kf_daily.resample('MS').std()
    
    # Strict Date Filter (before PCA)
    mask = (vol_monthly.index >= START_DATE) & (vol_monthly.index <= END_DATE)
    vol_monthly = vol_monthly.loc[mask]
    
    # Log Transform & PCA Orthogonalization
    vol_log = np.log(vol_monthly + 1e-6)
    
    pca = PCA(n_components=1)
    # Standardization before PCA to balance weights
    vol_log_std = StandardScaler().fit_transform(vol_log)
    common_factor = pca.fit_transform(vol_log_std)
    
    # Residuals (Orthogonalization)
    vol_ortho = pd.DataFrame(index=vol_log.index, columns=vol_log.columns)
    X_factor = sm.add_constant(common_factor)
    
    for col in vol_log.columns:
        vol_ortho[col] = sm.OLS(vol_log[col], X_factor).fit().resid
        
    # --- TEMPORAL ALIGNMENT ---
    # 1. Calculate AR(1) Innovations separately
    industry_innov = get_ar1_innovations(vol_ortho)
    theta_innov = get_ar1_innovations(theta)
    
    # 2. Strict intersection
    common_idx = industry_innov.index.intersection(theta_innov.index)
    industry_innov = industry_innov.loc[common_idx]
    theta_innov = theta_innov.loc[common_idx]
    
    print(f"Volatility alignment: {len(common_idx)} common months.")
    
    # Analysis
    # Comprehensive list of sectors from the paper
    TARGET_SECTORS = {
        'Automotive': ['Autos', 'Auto'], 
        'Banking': ['Banks', 'Bank'], 
        'Computers': ['Hrdwr', 'Hardw', 'Comps'], 
        'Oil': ['Oil', 'Oil  '],
        'Tobacco': ['Smoke'],
        'Pharmaceuticals': ['Drugs', 'Drug', 'Pharma'] # Robust addition
    }
    
    # Local stars function if not imported
    def local_get_stars(p):
        if p < 0.01: return "***"
        if p < 0.05: return "**"
        if p < 0.1: return "*"
        return ""
    
    for name, keywords in TARGET_SECTORS.items():
        # Find the correct column (Case-insensitive fuzzy search)
        # Search if a keyword is contained in the column name
        col_name = None
        for col in industry_innov.columns:
            clean_col = col.strip().lower()
            if any(k.lower() == clean_col for k in keywords):
                col_name = col
                break
        
        if col_name:
            # Lasso
            y_target = industry_innov[col_name]
            
            # Standardization on aligned data
            scaler = StandardScaler()
            X_s = pd.DataFrame(scaler.fit_transform(theta_innov), index=theta_innov.index, columns=theta_innov.columns)
            y_s = pd.Series(scaler.fit_transform(y_target.values.reshape(-1,1)).flatten(), index=y_target.index)
            
            # Selection and Model
            top_vars = select_exact_k_lars(X_s, y_s, k=5)
            model = sm.OLS(y_s, sm.add_constant(X_s[top_vars])).fit()
            
            # --- FORMALIZED DISPLAY ---
            print(f"\n{'-'*65}")
            print(f"TABLE 4: {name.upper()} ({col_name}) | R2 = {model.rsquared:.2f}")
            print(f"{'-'*65}")
            
            # Results table creation
            res = pd.DataFrame({
                'Coeff': model.params[top_vars],
                'Pval': model.pvalues[top_vars]
            })
            
            # Name mapping (String safety)
            res['Topic'] = [labels_map.get(str(v), v) if str(v) in labels_map else labels_map.get(v, v) for v in res.index]
            
            # Sort by absolute coefficient (Most important first)
            res = res.reindex(res['Coeff'].abs().sort_values(ascending=False).index)
            
            # Pretty print loop
            for idx, row in res.iterrows():
                t_name = row['Topic']
                coeff = row['Coeff']
                stars = local_get_stars(row['Pval'])
                
                # Formatting: Name aligned left (35 chars) | Coeff aligned right (6.2f) + Stars
                print(f"{t_name:<35} | {coeff:>6.2f} {stars}")
            print(f"{'-'*65}")
                
        else:
            print(f"Sector not found: {name} (Searched: {keywords})")
else:
    print("Ken French data not available.")

In [ ]:
print("SECTION 4.2.2: INDUSTRY VOLATILITY (Final Robust)")

# 1. Download & Prep
kf_daily = get_ken_french_49_daily()

if kf_daily is not None:
    # Calculate Monthly Volatility
    vol_monthly = kf_daily.resample('MS').std()
    
    # Strict Date Filter (before PCA)
    mask = (vol_monthly.index >= START_DATE) & (vol_monthly.index <= END_DATE)
    vol_monthly = vol_monthly.loc[mask]
    
    # Log Transform & PCA Orthogonalization
    vol_log = np.log(vol_monthly + 1e-6)
    
    pca = PCA(n_components=1)
    # Standardization before PCA to balance weights
    vol_log_std = StandardScaler().fit_transform(vol_log)
    common_factor = pca.fit_transform(vol_log_std)
    
    # Residuals (Orthogonalization)
    vol_ortho = pd.DataFrame(index=vol_log.index, columns=vol_log.columns)
    X_factor = sm.add_constant(common_factor)
    
    for col in vol_log.columns:
        vol_ortho[col] = sm.OLS(vol_log[col], X_factor).fit().resid
        
    # --- TEMPORAL ALIGNMENT ---
    # 1. Calculate AR(1) Innovations separately
    industry_innov = get_ar1_innovations(vol_ortho)
    theta_innov = get_ar1_innovations(theta)
    
    # 2. Strict intersection
    common_idx = industry_innov.index.intersection(theta_innov.index)
    industry_innov = industry_innov.loc[common_idx]
    theta_innov = theta_innov.loc[common_idx]
    
    print(f"Volatility alignment: {len(common_idx)} common months.")
    
    # Analysis
    # Comprehensive list of sectors from the paper
    TARGET_SECTORS = {
        'Automotive': ['Autos', 'Auto'], 
        'Banking': ['Banks', 'Bank'], 
        'Computers': ['Hrdwr', 'Hardw', 'Comps'], 
        'Oil': ['Oil', 'Oil  '],
        'Tobacco': ['Smoke'],
        'Pharmaceuticals': ['Drugs', 'Drug', 'Pharma'] 
    }
    
    # Local stars function
    def local_get_stars(p):
        if p < 0.01: return "***"
        if p < 0.05: return "**"
        if p < 0.1: return "*"
        return ""
    
    for name, keywords in TARGET_SECTORS.items():
        # Find the correct column (Case-insensitive fuzzy search)
        col_name = None
        for col in industry_innov.columns:
            clean_col = col.strip().lower()
            if any(k.lower() == clean_col for k in keywords):
                col_name = col
                break
        
        if col_name:
            # Lasso
            y_target = industry_innov[col_name]
            
            # Standardization on aligned data
            scaler = StandardScaler()
            X_s = pd.DataFrame(scaler.fit_transform(theta_innov), index=theta_innov.index, columns=theta_innov.columns)
            y_s = pd.Series(scaler.fit_transform(y_target.values.reshape(-1,1)).flatten(), index=y_target.index)
            
            # Lasso Selection (returns indices or names)
            top_vars = select_exact_k_lars(X_s, y_s, k=5)
            
            # --- CRITICAL FIX: Indices -> Names Conversion ---
            if len(top_vars) > 0 and isinstance(top_vars[0], (int, np.integer)):
                top_vars = X_s.columns[top_vars]
            # -------------------------------------------------
            
            # OLS Model
            model = sm.OLS(y_s, sm.add_constant(X_s[top_vars])).fit()
            
            # --- FORMALIZED DISPLAY ---
            print(f"\n{'-'*75}")
            print(f"TABLE 4: {name.upper()} ({col_name}) | R2 = {model.rsquared:.2f}")
            print(f"{'-'*75}")
            print(f"{'Topic':<35} | {'Coeff.':>7} | {'P-val':>8} | {'Sig'}")
            print(f"{'-'*75}")
            
            # Results table creation
            res = pd.DataFrame({
                'Coeff': model.params[top_vars],
                'Pval': model.pvalues[top_vars]
            })
            
            # Name mapping (String safety)
            res['Topic'] = [labels_map.get(str(v), v) if str(v) in labels_map else labels_map.get(v, v) for v in res.index]
            
            # Sort by absolute coefficient
            res = res.reindex(res['Coeff'].abs().sort_values(ascending=False).index)
            
            # Pretty print loop
            for idx, row in res.iterrows():
                t_name = row['Topic']
                coeff = row['Coeff']
                p_val = row['Pval']
                stars = local_get_stars(p_val)
                
                print(f"{t_name:<35} | {coeff:>7.2f} | {p_val:>8.4f} | {stars}")
            print(f"{'-'*75}")
                
        else:
            print(f"Sector not found: {name} (Searched: {keywords})")
else:
    print("Ken French data not available.")

##### 1.4. Economic policy uncertainty

In [ ]:
warnings.filterwarnings("ignore")

# End date of the paper
END_DATE_PAPER = '2017-06-01'

# 1. MAPPING CONFIGURATION
TARGET_MAP = {
    'Broad EPU': ['Economic Policy Uncertainty', 'News_Based_Policy_Uncert_Index'],
    'Entitlement Programs': ['Entitlement Programs', 'Entitlement'],
    'Financial Regulation': ['Financial Regulation', 'Financial'],
    'Fiscal Policy': ['Fiscal Policy', 'Fiscal'],
    'Government Spending': ['Government Spending'],
    'Health Care': ['Health Care', 'Healthcare'],
    'Monetary Policy': ['Monetary Policy'],
    'National Security': ['National Security'],
    'Regulation': ['Regulation'],
    'Sovereign Debt': ['Sovereign Debt'],
    'Taxes': ['Taxes', 'Tax'],
    'Trade Policy': ['Trade Policy']
}

# 2. DATA LOADING
# A. Topics
try:
    if 'theta' not in locals(): theta, labels_map = load_topics('data')
except: pass

# B. EPU Excel
file_path = 'data/Categorical_EPU_Data.xlsx'

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    try:
        epu_raw = pd.read_excel(file_path)
        
        # Date Cleaning
        epu_raw = epu_raw.dropna(subset=['Year', 'Month'])
        epu_raw['Date'] = pd.to_datetime(
            epu_raw['Year'].astype(int).astype(str) + '-' + 
            epu_raw['Month'].astype(int).astype(str) + '-01'
        )
        epu_raw = epu_raw.set_index('Date').sort_index()
        
        # Keep Data Columns
        cols_to_keep = [c for c in epu_raw.columns if c not in ['Year', 'Month']]
        epu_cats = epu_raw[cols_to_keep]
        
        # Log Transform
        epu_cats_log = np.log(epu_cats + 1)
        
        # Alignment
        common = theta.index.intersection(epu_cats_log.index)
        common = common[common <= END_DATE_PAPER]

        X_topics = theta.loc[common]
        Y_epu = epu_cats_log.loc[common]

        print(f"   -> Common period: {len(common)} months.")

        # 3. ANALYSIS
        def analyze_epu_category(cat_display_name, y_series):
            # Prep
            scaler_X = StandardScaler()
            scaler_y = StandardScaler()
            X_s = pd.DataFrame(scaler_X.fit_transform(X_topics), index=X_topics.index, columns=X_topics.columns)
            y_s = pd.Series(scaler_y.fit_transform(y_series.values.reshape(-1, 1)).flatten(), index=y_series.index)
            
            # Lasso Selection
            top_indices = select_exact_k_lars(X_s, y_s, k=5)
            
            # Indices vs names handling
            if len(top_indices) > 0 and isinstance(top_indices[0], (int, np.integer)):
                selected_vars = X_s.columns[top_indices]
            else:
                selected_vars = top_indices 
            
            if len(selected_vars) == 0: return 
            
            # OLS
            model = sm.OLS(y_s, sm.add_constant(X_s[selected_vars])).fit()
            
            # Print Header
            print(f"\n{'-'*75}") # Slightly longer line to accommodate p-val
            print(f"------------------- {cat_display_name.upper()} (R2 = {model.rsquared:.2f}) -------------------")
            print(f"{'-'*75}")
            print(f"{'Topic':<35} | {'Coeff':>6} | {'P-val':>8} | {'Sig'}")
            print(f"{'-'*75}")
            
            # Results DataFrame
            res = pd.DataFrame({
                'Coeff': model.params[selected_vars],
                'Pval': model.pvalues[selected_vars]
            })
            
            # Name Mapping
            res['Topic'] = [labels_map.get(str(v), v) if str(v) in labels_map else labels_map.get(v, v) for v in res.index]
            res = res.reindex(res['Coeff'].abs().sort_values(ascending=False).index)
            
            # Print Rows
            for _, row in res.iterrows():
                star = get_stars(row['Pval'])
                # ADD P-VALUE HERE
                print(f"{row['Topic']:<35} | {row['Coeff']:>6.2f} | {row['Pval']:>8.4f} | {star}")

        # Robust loop
        found_count = 0
        for display_name, search_keys in TARGET_MAP.items():
            found_col = None
            for col in Y_epu.columns:
                for key in search_keys:
                    if key.lower().strip() in col.lower().strip():
                        found_col = col
                        break
                if found_col: break
            
            if found_col:
                analyze_epu_category(display_name, Y_epu[found_col])
                found_count += 1
            else:
                print(f"Warning: Data for '{display_name}' not found.")

        print(f"\n{'-'*75}")
        print(f"Analyzed {found_count} categories successfully.")

    except Exception as e:
        print(f"Error processing EPU: {e}")

In [ ]:
warnings.filterwarnings("ignore")

# End date of the paper
END_DATE_PAPER = '2017-06-01'

# 1. MAPPING CONFIGURATION
TARGET_MAP = {
    'Broad EPU': ['Economic Policy Uncertainty', 'News_Based_Policy_Uncert_Index'],
    'Entitlement Programs': ['Entitlement Programs', 'Entitlement'],
    'Financial Regulation': ['Financial Regulation', 'Financial'],
    'Fiscal Policy': ['Fiscal Policy', 'Fiscal'],
    'Government Spending': ['Government Spending'],
    'Health Care': ['Health Care', 'Healthcare'],
    'Monetary Policy': ['Monetary Policy'],
    'National Security': ['National Security'],
    'Regulation': ['Regulation'],
    'Sovereign Debt': ['Sovereign Debt'],
    'Taxes': ['Taxes', 'Tax'],
    'Trade Policy': ['Trade Policy']
}

# 2. DATA LOADING
# A. Topics
try:
    if 'theta' not in locals(): theta, labels_map = load_topics('data')
except: pass

# B. EPU Excel
file_path = 'data/Categorical_EPU_Data.xlsx'

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    try:
        epu_raw = pd.read_excel(file_path)
        
        # Date Cleaning
        epu_raw = epu_raw.dropna(subset=['Year', 'Month'])
        epu_raw['Date'] = pd.to_datetime(
            epu_raw['Year'].astype(int).astype(str) + '-' + 
            epu_raw['Month'].astype(int).astype(str) + '-01'
        )
        epu_raw = epu_raw.set_index('Date').sort_index()
        
        # Keep Data Columns
        cols_to_keep = [c for c in epu_raw.columns if c not in ['Year', 'Month']]
        epu_cats = epu_raw[cols_to_keep]
        
        # Log Transform
        epu_cats_log = np.log(epu_cats + 1)
        
        # Alignment
        common = theta.index.intersection(epu_cats_log.index)
        common = common[common <= END_DATE_PAPER]

        X_topics = theta.loc[common]
        Y_epu = epu_cats_log.loc[common]

        print(f"   -> Common period: {len(common)} months.")

        # 3. ANALYSIS
        def analyze_epu_category(cat_display_name, y_series):
            # Prep
            scaler_X = StandardScaler()
            scaler_y = StandardScaler()
            X_s = pd.DataFrame(scaler_X.fit_transform(X_topics), index=X_topics.index, columns=X_topics.columns)
            y_s = pd.Series(scaler_y.fit_transform(y_series.values.reshape(-1, 1)).flatten(), index=y_series.index)
            
            # Lasso Selection
            top_indices = select_exact_k_lars(X_s, y_s, k=5)
            
            # Indices vs names handling
            if len(top_indices) > 0 and isinstance(top_indices[0], (int, np.integer)):
                selected_vars = X_s.columns[top_indices]
            else:
                selected_vars = top_indices 
            
            if len(selected_vars) == 0: return 
            
            # OLS
            model = sm.OLS(y_s, sm.add_constant(X_s[selected_vars])).fit()
            
            # Print Header
            print(f"\n{'-'*75}") # Slightly longer line to accommodate p-val
            print(f"------------------- {cat_display_name.upper()} (R2 = {model.rsquared:.2f}) -------------------")
            print(f"{'-'*75}")
            print(f"{'Topic':<35} | {'Coeff':>6} | {'P-val':>8} | {'Sig'}")
            print(f"{'-'*75}")
            
            # Results DataFrame
            res = pd.DataFrame({
                'Coeff': model.params[selected_vars],
                'Pval': model.pvalues[selected_vars]
            })
            
            # Name Mapping
            res['Topic'] = [labels_map.get(str(v), v) if str(v) in labels_map else labels_map.get(v, v) for v in res.index]
            res = res.reindex(res['Coeff'].abs().sort_values(ascending=False).index)
            
            # Print Rows
            for _, row in res.iterrows():
                star = get_stars(row['Pval'])
                # ADD P-VALUE HERE
                print(f"{row['Topic']:<35} | {row['Coeff']:>6.2f} | {row['Pval']:>8.4f} | {star}")

        # Robust loop
        found_count = 0
        for display_name, search_keys in TARGET_MAP.items():
            found_col = None
            for col in Y_epu.columns:
                for key in search_keys:
                    if key.lower().strip() in col.lower().strip():
                        found_col = col
                        break
                if found_col: break
            
            if found_col:
                analyze_epu_category(display_name, Y_epu[found_col])
                found_count += 1
            else:
                print(f"Warning: Data for '{display_name}' not found.")

        print(f"\n{'-'*75}")
        print(f"Analyzed {found_count} categories successfully.")

    except Exception as e:
        print(f"Error processing EPU: {e}")

#### Section 2. Emma